# FLUX.1 Krea Dev: 50 Denoising Capture

This notebook runs `black-forest-labs/FLUX.1-Krea-dev` in Colab, saves the final image, saves one latent tensor per denoising step, and then writes decoded latent preview PNGs.

Before running:

1. In Colab, choose **Runtime -> Change runtime type -> GPU**. A high-RAM A100/L4 runtime is preferred. T4 can be tight for 1024x1536.
2. Add a Colab Secret named `HF_TOKEN` using the key icon in the left sidebar, or paste the token when prompted.
3. Make sure the Hugging Face account behind the token has accepted access terms for `black-forest-labs/FLUX.1-Krea-dev`.

In [ ]:
# Install runtime dependencies. Colab already ships with torch, so avoid replacing it unless needed.
# This cell may restart the runtime once after repairing Pillow. After restart, run it again.
import importlib
import os
import shutil
import site
import subprocess
import sys
import sysconfig
from datetime import datetime, timezone
from pathlib import Path

MARKER = Path('/content/.flux_krea_deps_pillow_12_2_0_done') if Path('/content').exists() else Path('.flux_krea_deps_pillow_12_2_0_done')

DEPENDENCIES = [
    'diffusers>=0.35.0',
    'transformers>=4.44.0',
    'accelerate>=0.33.0',
    'huggingface-hub>=0.24.0',
    'sentencepiece>=0.2.0',
    'protobuf>=4.25.0',
    'safetensors>=0.4.0',
    'python-dotenv>=1.0.0',
]

def run_pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', *args])

def clear_pil_modules():
    for module_name in list(sys.modules):
        if module_name == 'PIL' or module_name.startswith('PIL.'):
            del sys.modules[module_name]
    importlib.invalidate_caches()

def site_package_roots():
    roots = set(site.getsitepackages())
    for key in ('purelib', 'platlib'):
        root = sysconfig.get_paths().get(key)
        if root:
            roots.add(root)
    return [Path(root) for root in sorted(roots) if root]

def remove_stale_pillow_files():
    for root_path in site_package_roots():
        for pattern in ('PIL', 'Pillow-*.dist-info', 'pillow-*.dist-info'):
            for path in root_path.glob(pattern):
                print('removing', path)
                shutil.rmtree(path, ignore_errors=True)

def verify_pillow():
    clear_pil_modules()
    import PIL, PIL._typing, PIL.ImageText
    print('Pillow', PIL.__version__, PIL.__file__)
    assert hasattr(PIL._typing, '_Ink'), 'PIL._typing._Ink is missing.'
    print('Pillow _Ink check: ok')

def restart_runtime():
    print('Dependencies installed. Restarting runtime to unload any old Pillow binary extension.')
    print('After the restart, rerun this cell once; it should print "Pillow _Ink check: ok" and continue.')
    try:
        from google.colab import runtime
        runtime.restart_runtime()
    except Exception:
        os.kill(os.getpid(), 9)

needs_install = not MARKER.exists()
if not needs_install:
    try:
        verify_pillow()
    except Exception as exc:
        print('Existing dependency marker is stale:', repr(exc))
        MARKER.unlink(missing_ok=True)
        needs_install = True

if needs_install:
    clear_pil_modules()
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'Pillow'], check=False)
    remove_stale_pillow_files()
    run_pip('install', '--no-cache-dir', '--force-reinstall', 'Pillow==12.2.0')
    run_pip('install', '-U', *DEPENDENCIES)
    MARKER.write_text(datetime.now(timezone.utc).isoformat() + '\n', encoding='utf-8')
    restart_runtime()
else:
    import diffusers, transformers, accelerate, huggingface_hub
    print('diffusers', diffusers.__version__)
    print('transformers', transformers.__version__)
    print('accelerate', accelerate.__version__)
    print('huggingface_hub', huggingface_hub.__version__)


In [ ]:
import gc
import json
import math
import os
import shutil
import sys
import time
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import torch
from huggingface_hub import HfApi, login

# Drop any stale in-memory PIL modules left over from a previous failed Colab import.
for module_name in list(sys.modules):
    if module_name == 'PIL' or module_name.startswith('PIL.'):
        del sys.modules[module_name]

# Optional: if you upload a .env file to Colab, this will read HF_TOKEN from it.
try:
    from dotenv import load_dotenv
    for dotenv_path in (Path('.env'), Path('/content/.env')):
        if dotenv_path.exists():
            load_dotenv(dotenv_path)
except Exception:
    pass

# Temporary Colab bypass: paste your raw hf_... token here if Colab Secrets are stale.
# Leave empty to use env/.env/Colab Secrets/password prompt.
HARDCODED_HF_TOKEN = '*******'

def normalize_hf_token(token):
    if not token:
        return None
    token = str(token).strip().strip('"').strip("'")
    if token.lower().startswith('bearer '):
        token = token.split(None, 1)[1].strip()
    return token or None

def token_candidates():
    token = normalize_hf_token(HARDCODED_HF_TOKEN)
    if token:
        yield 'HARDCODED_HF_TOKEN notebook variable', token
    for name in ('HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
        token = normalize_hf_token(os.environ.get(name))
        if token:
            yield f'environment variable {name}', token
    try:
        from google.colab import userdata
        token = normalize_hf_token(userdata.get('HF_TOKEN'))
        if token:
            yield 'Colab Secret HF_TOKEN', token
    except Exception:
        pass

def validate_hf_token(source, token):
    try:
        user = HfApi().whoami(token=token)
        print(f'Using Hugging Face token from {source} for user: {user.get("name") or user.get("fullname") or "<unknown>"}')
        return True
    except Exception as exc:
        print(f'Rejected Hugging Face token from {source}: {type(exc).__name__}')
        return False

def get_hf_token():
    for source, token in token_candidates():
        if validate_hf_token(source, token):
            return token
    while True:
        token = normalize_hf_token(getpass('Paste Hugging Face token with FLUX.1-Krea-dev access: '))
        if token and validate_hf_token('password prompt', token):
            return token
        print('That token was not accepted by Hugging Face. Create or update a read token at https://huggingface.co/settings/tokens and try again.')

HF_TOKEN = get_hf_token()
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'gpu memory: {total_gb:.1f} GB')
else:
    raise RuntimeError('Switch the Colab runtime to GPU before running FLUX.')

## Run Configuration

The target project settings are 50 steps at 1024x1536. If Colab runs out of memory, keep `STEPS = 50` and reduce only `WIDTH`/`HEIGHT` to another 2:3 size such as 768x1152.

In [ ]:
MODEL_ID = 'black-forest-labs/FLUX.1-Krea-dev'

PROMPT = """hyperreal close portrait of a woman meeting the viewer's gaze head-on, eyes steady and lucid beneath a helmet cropped tight at the frame's edge. her face carries a dusting of white frost along the lashes and temples, lips slightly parted as if mid-breath, fine moisture glinting where the cold meets warmth. the light comes from behind a low golden sun cutting through icy air wrapping her in a soft halo while highlights cling delicately to the frost edges. her skin reveals honest micro detail: faint pores, subsurface glow on the cheeks, and a natural sheen from the chill. the palette moves between warm amber and arctic blue. every surface behaves realistically matte skin diffusing light, frost refracting it, the atmosphere crisp yet breathable. the mood is tender intensity, a quiet warmth radiating through the ice. captured on an 85mm lens at f/2.0, focus locked to her eyes, shallow depth isolating her face in luminous realism.
--v 7 --ar 2:3 --raw --profile"""

SEED = 0
WIDTH = 1024
HEIGHT = 1536
STEPS = 50
GUIDANCE_SCALE = 4.5
MAX_SEQUENCE_LENGTH = 512

# Use CPU offload on smaller Colab GPUs. Set False on A100/high-memory runtimes for speed.
USE_CPU_OFFLOAD = True

# Save VAE-decoded previews after generation. If this fails, normalized previews are still written.
DECODE_LATENT_PREVIEWS = True

# Set True to write artifacts directly into Google Drive.
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/flux_krea_runs')
else:
    OUTPUT_ROOT = Path('/content/flux_krea_runs')

RUN_ID = f"flux_krea_colab_seed{SEED}_{WIDTH}x{HEIGHT}_{STEPS}steps_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_DIR = OUTPUT_ROOT / RUN_ID
LATENTS_DIR = RUN_DIR / 'latents'
DECODED_DIR = RUN_DIR / 'decoded_latents'
FINAL_DIR = RUN_DIR / 'final image'
for path in (LATENTS_DIR, DECODED_DIR, FINAL_DIR):
    path.mkdir(parents=True, exist_ok=True)

(RUN_DIR / 'prompt.txt').write_text(PROMPT + '\n', encoding='utf-8')
print('run dir:', RUN_DIR)

In [ ]:
from diffusers import FluxPipeline

def choose_dtype():
    major, _minor = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

dtype = choose_dtype()
print('loading pipeline with dtype:', dtype)

pipe = FluxPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    token=HF_TOKEN,
)

if hasattr(pipe, 'vae'):
    if hasattr(pipe.vae, 'enable_tiling'):
        pipe.vae.enable_tiling()
    if hasattr(pipe.vae, 'enable_slicing'):
        pipe.vae.enable_slicing()

if USE_CPU_OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to('cuda')

print('pipeline ready')

In [ ]:

latent_records = []

def capture_latents(pipe, step, timestep, callback_kwargs):
    latents = callback_kwargs.get('latents')
    if latents is None:
        return callback_kwargs
    latent_cpu = latents.detach().cpu().clone()
    latent_path = LATENTS_DIR / f'step_{int(step):03d}.pt'
    torch.save(latent_cpu, latent_path)
    latent_records.append({
        'step': int(step),
        'timestep': int(timestep) if hasattr(timestep, '__int__') else str(timestep),
        'latents_path': str(latent_path),
        'latents_shape': list(latent_cpu.shape),
        'latents_dtype': str(latent_cpu.dtype),
    })
    return callback_kwargs

def write_metadata_update(**updates):
    metadata.update(updates)
    (RUN_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

metadata = {
    'status': 'initialized',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'execution_mode': 'colab_local_diffusers',
    'model_id': MODEL_ID,
    'prompt': PROMPT,
    'seed': SEED,
    'width': WIDTH,
    'height': HEIGHT,
    'num_inference_steps': STEPS,
    'guidance_scale': GUIDANCE_SCALE,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'torch_dtype': str(dtype),
    'use_cpu_offload': USE_CPU_OFFLOAD,
    'output': {
        'run_dir': str(RUN_DIR),
        'prompt_path': str(RUN_DIR / 'prompt.txt'),
        'metadata_path': str(RUN_DIR / 'metadata.json'),
        'latents_dir': str(LATENTS_DIR),
        'decoded_latents_dir': str(DECODED_DIR),
        'final_image_path': str(FINAL_DIR / 'image.png'),
    },
}
write_metadata_update()

generator_device = 'cpu' if USE_CPU_OFFLOAD else 'cuda'
generator = torch.Generator(device=generator_device).manual_seed(SEED)

started = time.time()
try:
    with torch.inference_mode():
        result = pipe(
            prompt=PROMPT,
            height=HEIGHT,
            width=WIDTH,
            guidance_scale=GUIDANCE_SCALE,
            num_inference_steps=STEPS,
            num_images_per_prompt=1,
            max_sequence_length=MAX_SEQUENCE_LENGTH,
            generator=generator,
            callback_on_step_end=capture_latents,
            callback_on_step_end_tensor_inputs=['latents'],
        )

    final_image = result.images[0]
    final_path = FINAL_DIR / 'image.png'
    final_image.save(final_path, format='PNG')
except Exception as exc:
    write_metadata_update(
        status='generation_failed',
        duration_seconds_generation=round(time.time() - started, 3),
        inference_attempted=True,
        error=f'{type(exc).__name__}: {exc}',
        latents_saved=latent_records,
        latent_file_count=len(list(LATENTS_DIR.glob('step_*.pt'))),
    )
    print('generation failed after saving', metadata['latent_file_count'], 'latent tensors')
    print('error:', repr(exc))
    raise
else:
    write_metadata_update(
        status='generated',
        duration_seconds_generation=round(time.time() - started, 3),
        inference_attempted=True,
        inference_ran=True,
        final_image_path=str(final_path),
        latents_saved=latent_records,
        latent_file_count=len(list(LATENTS_DIR.glob('step_*.pt'))),
    )

    print('saved final image:', final_path)
    print('saved latent tensors:', metadata['latent_file_count'])


In [ ]:

from PIL import Image
from diffusers import AutoencoderKL
from diffusers.image_processor import VaeImageProcessor
from diffusers.pipelines.flux.pipeline_flux import FluxPipeline

VAE_SCALE_FACTOR = int(getattr(pipe, 'vae_scale_factor', 8))
DECODE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DECODE_DTYPE = torch.float32
_decode_vae = None
_image_processor = VaeImageProcessor(vae_scale_factor=VAE_SCALE_FACTOR)

def load_decode_vae():
    global _decode_vae
    if _decode_vae is None:
        print('loading clean VAE for latent decoding on', DECODE_DEVICE)
        _decode_vae = AutoencoderKL.from_pretrained(
            MODEL_ID,
            subfolder='vae',
            torch_dtype=DECODE_DTYPE,
            token=HF_TOKEN,
        )
        if hasattr(_decode_vae, 'enable_tiling'):
            _decode_vae.enable_tiling()
        if hasattr(_decode_vae, 'enable_slicing'):
            _decode_vae.enable_slicing()
        _decode_vae.to(DECODE_DEVICE)
        _decode_vae.eval()
    return _decode_vae

def unpack_flux_latents_if_needed(latents, height, width):
    if latents.ndim != 3:
        return latents
    try:
        return FluxPipeline._unpack_latents(latents, height, width, VAE_SCALE_FACTOR)
    except Exception:
        batch, patches, channels = latents.shape
        if channels % 4 != 0:
            return latents
        latent_h = 2 * (height // (VAE_SCALE_FACTOR * 2))
        latent_w = 2 * (width // (VAE_SCALE_FACTOR * 2))
        expected = (latent_h // 2) * (latent_w // 2)
        if expected != patches:
            return latents
        return latents.view(batch, latent_h // 2, latent_w // 2, channels // 4, 2, 2).permute(0, 3, 1, 4, 2, 5).reshape(batch, channels // 4, latent_h, latent_w)

def prepare_for_vae(latents, vae, height, width):
    latents = unpack_flux_latents_if_needed(latents.detach().float(), height, width)
    config = vae.config
    scaling_factor = getattr(config, 'scaling_factor', None)
    shift_factor = getattr(config, 'shift_factor', None)
    if scaling_factor not in (None, 0):
        latents = latents / scaling_factor
    if shift_factor is not None:
        latents = latents + shift_factor
    return latents

def normalized_preview(latents):
    x = latents.detach().float().cpu()
    if x.ndim == 3:
        x = x[0].T
        side_h = int(math.sqrt(x.shape[1]))
        side_w = max(1, x.shape[1] // max(1, side_h))
        x = x[:, :side_h * side_w].reshape(x.shape[0], side_h, side_w)
    elif x.ndim == 4:
        x = x[0]
    else:
        x = x.reshape(1, 1, -1)
    x = x[:3]
    if x.shape[0] == 1:
        x = x.repeat(3, 1, 1)
    elif x.shape[0] == 2:
        x = torch.cat([x, x[:1]], dim=0)
    x_min, x_max = x.min(), x.max()
    x = (x - x_min) / (x_max - x_min + 1e-8)
    array = (x.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype('uint8')
    return Image.fromarray(array, mode='RGB')

def save_decoded_preview(latent_path, output_path):
    latents = torch.load(latent_path, map_location='cpu')
    if not DECODE_LATENT_PREVIEWS:
        normalized_preview(latents).save(output_path, format='PNG')
        return 'normalized_disabled', None
    try:
        vae = load_decode_vae()
        prepared = prepare_for_vae(latents, vae, HEIGHT, WIDTH).to(device=DECODE_DEVICE, dtype=DECODE_DTYPE)
        with torch.inference_mode():
            decoded = vae.decode(prepared, return_dict=False)[0]
        image = _image_processor.postprocess(decoded.detach().cpu(), output_type='pil')[0]
        image.save(output_path, format='PNG')
        return 'vae', None
    except Exception as exc:
        normalized_preview(latents).save(output_path, format='PNG')
        return f'normalized_fallback:{type(exc).__name__}', f'{type(exc).__name__}: {exc}'

decode_started = time.time()
preview_records = []
for latent_path in sorted(LATENTS_DIR.glob('step_*.pt')):
    output_path = DECODED_DIR / (latent_path.stem + '.png')
    mode, error = save_decoded_preview(latent_path, output_path)
    record = {'step': int(latent_path.stem.split('_')[-1]), 'path': str(output_path), 'mode': mode}
    if error:
        record['error'] = error
    preview_records.append(record)
    print(f"{latent_path.name} -> {output_path.name} ({mode})")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metadata_path = RUN_DIR / 'metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
metadata.update({
    'status': 'decoded_latents_complete' if metadata.get('status') == 'generation_failed' else 'complete',
    'duration_seconds_decode_previews': round(time.time() - decode_started, 3),
    'decoded_latents_saved': preview_records,
    'decoded_latent_file_count': len(list(DECODED_DIR.glob('step_*.png'))),
})
if any(record['mode'].startswith('normalized_fallback') for record in preview_records):
    metadata['decoded_latents_warning'] = 'One or more latent previews are normalized tensor visualizations, not VAE-decoded images.'
metadata_path.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

print('decoded previews:', metadata['decoded_latent_file_count'])


In [ ]:

latent_count = len(list(LATENTS_DIR.glob('step_*.pt')))
preview_paths = sorted(DECODED_DIR.glob('step_*.png'))
preview_count = len(preview_paths)
final_path = FINAL_DIR / 'image.png'
final_exists = final_path.exists()
metadata_path = RUN_DIR / 'metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
preview_records = metadata.get('decoded_latents_saved', [])
last_preview_record = next((record for record in preview_records if record.get('step') == STEPS - 1), None)

# Only recover the final image from the final latent if that preview was genuinely VAE-decoded.
# Normalized fallback previews are useful diagnostics, but they are not images from the model.
if not final_exists and last_preview_record and last_preview_record.get('mode') == 'vae':
    recovered_path = Path(last_preview_record['path'])
    if recovered_path.exists():
        shutil.copy2(recovered_path, final_path)
        final_exists = final_path.exists()
        metadata['final_image_path'] = str(final_path)
        metadata['final_image_recovered_from_vae_latent'] = str(recovered_path)
        metadata_path.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
elif not final_exists:
    metadata['final_image_missing_reason'] = 'No real final image was saved, and the last latent preview was not VAE-decoded.'
    metadata_path.write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

print('run dir:', RUN_DIR)
print('final image exists:', final_exists)
print('latent count:', latent_count)
print('decoded preview count:', preview_count)

assert final_exists, 'Final image is missing. The notebook did not replace it with a normalized latent preview.'
assert latent_count == STEPS, f'Expected {STEPS} latent tensors, found {latent_count}.'
assert preview_count == STEPS, f'Expected {STEPS} decoded previews, found {preview_count}.'

from IPython.display import display
display(Image.open(final_path))


In [ ]:
# Zip the run artifacts for download or Drive retention.
zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('zip:', zip_path)

# Uncomment to trigger a browser download in Colab.
# from google.colab import files
# files.download(zip_path)